In [1]:
import pdfplumber
import pandas as pd

pdf_path = "data/Variables_overview_VALUE_Marketdata.pdf"

rows = []

table_settings = {
    "text_x_tolerance": 1,
    "text_y_tolerance": 3
}

with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables(table_settings=table_settings)

        for table in tables:
            for row in table:
                rows.append(row)

combined = pd.DataFrame(rows)
with pd.option_context(
    "display.max_rows", None,
    #"display.max_colwidth", None
):
    display(combined)

,0,1,2,3,4
0,angebot_id,-,ID,Unique ID within the Value market databasee.,offer_id
1,oadr_koord_lat_epsg4326,latlon,Latitude,The latitude of the WGS84 coordinate (EPSG 4326).,addr_coord_lat
2,oadr_koord_genauigkeit,integer-positive,Accuracy of coordinate,The accuracy of the object coordinate (radius ...,addr_coord_accuracy
3,oadr_strasse,text,Street Number Annex,"The postal address (street, house number and a...",addr_street_number
4,oadr_u2,text,Subdivision 2nd level (U2),The second administrative subdivision of the m...,-
5,oadr_ortsteil,text,Locality,The locality (e.g. urban district or local sub...,-
6,oadr_plz,postal code,Postcode,The postcode.,code_postal
7,oadr_gkz,integer-positive,Municipality code,The official municipality code (Destatis).,code_municipality
8,oadr_kreis,text,Administrative district,The name of the administrative district (Desta...,-
9,oadr_bl,integer-positive,Federal state,The federal state key.,-


In [2]:
import pandas as pd

# Read saved NaN statistics
nan_stats = pd.read_csv(
    "nan_stats.csv",
    index_col=0
)

nan_stats.index.name = "feature"


# Prepare descriptions extracted from PDF
pdf_info = combined.copy()

pdf_info.columns = [
    "feature",
    "original_type",
    "description",
    "pdf_info_3",
    "pdf_info_4"
]

# Clean feature names
pdf_info["feature"] = (
    pdf_info["feature"]
    .astype(str)
    .str.strip()
)

nan_stats.index = (
    nan_stats.index
    .astype(str)
    .str.strip()
)

# Remove duplicate feature definitions, if any
pdf_info = pdf_info.drop_duplicates(
    subset="feature",
    keep="first"
)

# Use feature as index for joining
pdf_info = pdf_info.set_index("feature")


# Join PDF information to nan_stats
nan_stats = nan_stats.join(
    pdf_info,
    how="left"
)

In [3]:
# 4. Display
with pd.option_context(
    "display.max_rows", None,
):
    display(nan_stats)

,dtype,missing_count,missing_percent,unique_count,values,original_type,description,pdf_info_3,pdf_info_4
feature,,,,,,,,,
anzart_wohnen_auf_zeit_janein,float64,144778,100.00,0,NaN,NaN,NaN,NaN,NaN
anzart_waz_moebliert_janein,float64,144778,100.00,0,NaN,truefalse,Advertisement type temporary\naccomodation/fur...,This is the offer of an accomodation object th...,-
oadr_ortz_reg,float64,144778,100.00,0,NaN,NaN,NaN,NaN,NaN
anz_parken_parkhaus,float64,144768,99.99,3,NaN,NaN,NaN,NaN,NaN
anz_gebaeude,float64,144511,99.82,4,NaN,integer-positive,Number of buildings,The number of buildings.,-
anz_einheiten_gewerbe,float64,144447,99.77,11,NaN,integer-positive,Commercial units offered,The number of offered commercial units in the ...,-
anz_einheiten_gewerbe_gesamt,float64,144362,99.71,21,NaN,NaN,NaN,NaN,NaN
anz_parken_carports,float64,143186,98.90,4,NaN,NaN,NaN,NaN,NaN
oeig_flachdach_janein,object,138458,95.63,2,"[False, True]",NaN,NaN,NaN,NaN


In [4]:
nan_stats = nan_stats.drop(columns=["values"])
nan_stats.to_csv("nan_stats.csv", index=True)